Появилась идея через geopy находить широту и долготу по адресу вакансии для дальнейшего построения карты https://habr.com/ru/companies/otus/articles/760148/

In [1]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from time import sleep
import pandas as pd
import re

def clean_address(address):
    if pd.isna(address):
        return None
    
    #регулярки для чистки ненужных обозначений и лишней информации
    address = re.sub(r'\b(метро|д|р-н|район|г|поселок|деревня|тер|пгт|село|мкр|влд)\.?\b', '', address)
    address = re.sub(r'\b(стр|к|помещ|кв|м|литера|офис)\b.*$', '', address)
    address = re.sub(r'\([^)]*\)', '', address)
    
    #заменяем сокращения
    replacements = {
        'б-р': 'бульвар',
        'пр-кт': 'проспект',
        'пр-т': 'проспект',
    }
    for short, full in replacements.items():
        address = address.replace(short, full)
    
    #обрезаем строки
    if 'москва' in address.lower():
        pos = address.lower().rfind('москва')
        address = address[pos:]
    if 'санкт-петербург' in address.lower():
        pos = address.lower().rfind('санкт-петербург')
        address = address[pos:]
    if 'екатеринбург' in address.lower():
        pos = address.lower().rfind('екатеринбург')
        address = address[pos:]

    #финальная чистка
    address = re.sub(r'\s+', ' ', address)
    address = address.strip(' ,.')
    
    return address

geolocator = Nominatim(user_agent="rabota_ru_lol", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)


def get_lat_lon(address):
    address = clean_address(address)
    loc = geocode(address)
    if loc:
        return loc.latitude, loc.longitude
    else:
        #иногда нужно убрать ул, чтобы адрес нашелся
        address = re.sub(r'\b(ул)\.?\b', '', address)
        loc = geocode(address)
        if loc:
            return loc.latitude, loc.longitude
        print(f"Адрес: {address}. Не найден")
        return None, None

In [ ]:
df_rr = pd.read_csv('../datasets/rabota_ru_formated.csv')
c = 0
for i, r in df_rr.iterrows():
    lat, lon = get_lat_lon(r['Адрес'])
    df_rr.loc[i, 'Широта'] = lat
    df_rr.loc[i, 'Долгота'] = lon
    
    c += 1  
    if c % 20 == 0:
        df_rr.to_csv('../datasets/rabota_ru_final.csv', index=False)
        print(f'Сохранено {c} строк')

df_rr.to_csv('../datasets/rabota_ru_final.csv', index=False)

Сохранено 20 строк
Сохранено 40 строк
Сохранено 60 строк
Сохранено 80 строк
Сохранено 100 строк
Сохранено 120 строк
Адрес: Москва, проезд Проектируемый 5112-й, 6. Не найден
Сохранено 140 строк
Сохранено 160 строк
Сохранено 180 строк
Адрес: Москва,  Дмитровка Б., 11. Не найден
Адрес: Москва,  Дмитровка Б., 11. Не найден
Сохранено 200 строк
Адрес: Москва, МКАД, 47-й км, 3. Не найден
Адрес: Москва,  16-я Парковая, 26. Не найден
Сохранено 220 строк
Сохранено 240 строк
Сохранено 260 строк
Адрес: Москва,  1-я Владимирская, 10Д. Не найден
Адрес: Москва, поселение Сосенское, Николо-Хованское, 1014. Не найден
Сохранено 280 строк
Сохранено 300 строк
Адрес: Москва, поселение Сосенское, Коммунарка,  Потаповская Роща, 3. Не найден
Сохранено 320 строк
Адрес: Москва, проезд 1-й Перова Поля, 3. Не найден
Адрес: Москва, проезд 1-й Перова Поля, 3. Не найден
Сохранено 340 строк
Сохранено 360 строк
Адрес: Москва,  Большая Почтовая, 26В. Не найден
Сохранено 380 строк
Адрес: Москва,  16-я Парковая, 30. Не н

In [6]:
df_rr['Долгота'].isna().sum()

np.int64(1120)

После первого прохождения получилось определить > 90% адресов. Посмотрев на ненайденные адреса можно определить еще несколько проблем. основная - проблемы с пониманием окончаний у чисел по типу "Москва,  15-я Парковая, 10", "Москва, МКАД 33-й км, 6". Также нужно убрать упоминания поселений.

In [ ]:
def clean_address(address):
    if pd.isna(address):
        return None
    #регулярки для чистки ненужных обозначений и лишней информации
    address = re.sub(r'\b(метро|д|р-н|район|г|поселок|деревня|тер|пгт|село|мкр|влд|двлд)\.?\b', '', address)
    address = re.sub(r'\b(стр|к|помещ|кв|м|литера|офис|эт|оф)\b.*$', '', address)

    #убирает поселение и слово после него
    address = re.sub(r'\b(поселение)\b\s+[^\s,\.]+', '', address)
    address = re.sub(r'(\d+)-[яйое]+', r'\1', address)
    address = re.sub(r'\bкм\b', 'километр', address)
    address = re.sub(r'\([^)]*\)', '', address)
    
    #заменяем сокращения
    replacements = {
        'б-р': 'бульвар',
        'пр-кт': 'проспект',
        'пр-т': 'проспект',
    }
    for short, full in replacements.items():
        address = address.replace(short, full)
    
    #обрезаем строки
    if 'москва' in address.lower():
        pos = address.lower().rfind('москва')
        address = address[pos:]
    if 'санкт-петербург' in address.lower():
        pos = address.lower().rfind('санкт-петербург')
        address = address[pos:]
    if 'екатеринбург' in address.lower():
        pos = address.lower().rfind('екатеринбург')
        address = address[pos:]
    if 'московская обл' in address.lower():
        pos = address.lower().rfind('московская обл')
        address = address[pos:]

    #финальная чистка
    address = re.sub(r'\s+', ' ', address)
    address = address.strip(' ,.')
    
    return address

geolocator = Nominatim(user_agent="rabota_ru_lol", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

In [16]:
c = 0
for i, r in df_rr[df_rr['Долгота'].isna()].iterrows():
    lat, lon = get_lat_lon(r['Адрес'])
    df_rr.loc[i, 'Широта'] = lat
    df_rr.loc[i, 'Долгота'] = lon
    
    c += 1  
    if c % 20 == 0:
        df_rr.to_csv('../datasets/rabota_ru_final.csv', index=False)
        print(f'Сохранено {c} строк')

df_rr.to_csv('../datasets/rabota_ru_final.csv', index=False)

Адрес: Москва,  Дмитровка Б., 11. Не найден
Адрес: Москва,  Дмитровка Б., 11. Не найден
Адрес: Москва, МКАД, 47 километр, 3. Не найден
Адрес: Москва,  1 Владимирская, 10Д. Не найден
Адрес: Москва, проезд 1 Перова Поля, 3. Не найден
Адрес: Москва, проезд 1 Перова Поля, 3. Не найден
Адрес: Москва,  Большая Почтовая, 26В. Не найден
Адрес: Москва, МКАД, 47 километр, 3. Не найден
Адрес: Московская обл, Одинцово, Зайцево, Кокошкинское шоссе, 12. Не найден
Адрес: Москва, , Калужское шоссе, 22 километр, 10. Не найден
Адрес: Москва, улица Мастеркова, дом 4, 1 этаж, павильон № 11. Не найден
Адрес: Москва, Киевское шоссе, 22 километр, 4. Не найден
Сохранено 20 строк
Адрес: Москва, проезд 17 Марьиной Рощи, 9А. Не найден
Адрес: Москва, Пятницкое шоссе, 7км. Не найден
Адрес: Москва, Коммунарка, Калужское шоссе, 21 километр. Не найден
Адрес: Московская обл, Одинцово, Барвиха, 42. Не найден
Адрес: Москва, Солнцево, Киевское шоссе. Не найден
Адрес: Московская обл, Одинцово, Усово-Тупик. Не найден
Адрес

In [7]:
print(f'Всего осталось неопределенных адресов: {df_rr["Долгота"].isna().sum()}, Процент от всего датасета: {df_rr["Долгота"].isna().sum() / len(df_rr) * 100:.2f}')

Всего осталось неопределенных адресов: 861, Процент от всего датасета: 7.44


В итоге удалось определить еще некоторое количество адресов, в основном в Москве и Питере. Наибольшие проблемы с определением возникают в московской области и выбросах в других регионах, которых не должно быть в датасете. Но такие адреса вряд ли получится очистить с помощью регулярных выражений, тут нужен более сложный подход. Но даже так удалось добиться очень хорошего качества определения.